In [3]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC
import librosa
import torch

In [4]:
# processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")

In [5]:
# def load_wav(filename):
#     audio, sample_rate = sf.read(filename)
#     if sample_rate != 16000:
#         raise ValueError("Sample rate must be 16kHz and mono audio")
#     return audio

In [6]:
# def preprocess(file_path, label):
#     audio = load_wav(file_path.numpy().decode())
#     inputs = processor(audio, sampling_rate=16000, return_tensors="tf", padding="longest")
#     return inputs.input_values[0], label

In [7]:
# def tf_preprocess(file_path, label):
#     result = tf.py_function(preprocess, inp=[file_path, label], Tout=(tf.float32, tf.int64))
#     return {"input_values": result[0]}, result[1]

In [8]:
# data_path = os.listdir('data/')
# data_path

In [9]:
# x,y = [],[]

# for i,path in enumerate(data_path):
#     for file in os.listdir(f'data/{path}/'):
#         x.append(file)
#         y.append(i)

In [10]:
# classes = {
#     0 : 'Angry',
#     1 : 'Disgusted',
#     2 : 'Fearful',
#     3 : 'Happy',
#     4 : 'Sad',
# }

In [11]:
#dataset = tf.data.Dataset.from_tensor_slices((x, y))
#dataset = dataset.map(tf_preprocess).batch(4)

In [12]:
# from transformers import TFAutoModelForSequenceClassification

# model = TFAutoModelForSequenceClassification.from_pretrained(
#     "facebook/wav2vec2-base", num_labels=5
# )

In [13]:
from transformers import  AutoFeatureExtractor, AutoModelForAudioClassification

In [14]:
model_name = "superb/hubert-large-superb-er"
feature_extractor =  AutoFeatureExtractor.from_pretrained(model_name)
model = AutoModelForAudioClassification.from_pretrained(model_name)

e:\Desktop\Job\Final Year Project\Computer Vision\backend\v-env\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PCR\.cache\huggingface\hub\models--superb--hubert-large-superb-er. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [15]:
def predict_emotion(audio_path):
    audio, rate = librosa.load(audio_path, sr=16000)
    inputs = feature_extractor(audio, sampling_rate=rate, return_tensors="pt", padding=True)
    
    with torch.no_grad():
        outputs = model(inputs.input_values)
        predictions = torch.nn.functional.softmax(outputs.logits.mean(dim=1), dim=-1)
        predicted_label = torch.argmax(predictions, dim=-1)
        emotion = model.config.id2label[predicted_label.item()]
    return emotion

In [16]:

model.config.id2label

{0: 'neu', 1: 'hap', 2: 'ang', 3: 'sad'}

In [17]:
import os
angry_wav = os.path.join('data','Angry')
data_path = os.listdir(angry_wav)
data_path[0]

'03-01-05-01-01-01-01.wav'

In [18]:
import torchaudio

In [26]:
def predict_emotion(path):
    speech_array, sampling_rate = torchaudio.load(path)

    resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
    speech = resampler(speech_array).squeeze().numpy()

    inputs = feature_extractor(speech, sampling_rate=16000, return_tensors="pt", padding=True)

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_class_id = torch.argmax(logits).item()
    return model.config.id2label[predicted_class_id]
# predicted_label = model.config.id2label[predicted_class_id]

# predicted_label

In [27]:
print(predict_emotion(f"{angry_wav}/{data_path[0]}"))

ang


In [28]:
model.save_pretrained('audiomodel')
feature_extractor.save_pretrained('audiomodel')

['audiomodel\\preprocessor_config.json']

In [20]:
model.config.id2label

{0: 'neu', 1: 'hap', 2: 'ang', 3: 'sad'}

In [21]:
len(inputs.input_values[0])

61929

In [22]:
model.config.num_labels

4

In [23]:
outputs.logits.shape

NameError: name 'outputs' is not defined

In [ ]:
emotion = predict_emotion(f"{angry_wav}/{data_path[0]}")
print(f"Predicted emotion: {emotion}")

KeyError: 32